In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import shap 
import json

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA
import torch.nn.functional as torch_F # avoid import problems becasue F is used as a variable

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.mps.is_available(): device = torch.device("mps")
device

/Users/marekschuster/Documents/Studium/Thesis/AURA/x-med/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='mps')

In [2]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)

notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')

full_dataset = NephroCAGEDataset(static_df=static_df, ts_data=ts_data, notes_df=notes, biopsy_df=dfs['biopsy'])
datapoints_limit = len(full_dataset) # Can be set to low value like 10 for quick testing
dataset = Subset(full_dataset, indices=list(range(datapoints_limit)))
ts_scaler = full_dataset.ts_scaler
static_scaler = full_dataset.scaler

Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9
Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904
Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343
Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9


In [3]:
# Split dataset into training and test sets (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
print(test_size)

batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=full_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)

state_dict = torch.load('../models/after10epochs.pt', weights_only=True, map_location=torch.device('cpu'))
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Missing:", missing)
print("Unexpected:", unexpected)

model.eval()
model.to(device)

677
Missing: ['lstm_encoder.lstm.lstm_cells.0.W_decomp', 'lstm_encoder.lstm.lstm_cells.0.b_decomp', 'lstm_encoder.lstm.lstm_cells.0.Wi', 'lstm_encoder.lstm.lstm_cells.0.Ui', 'lstm_encoder.lstm.lstm_cells.0.bi', 'lstm_encoder.lstm.lstm_cells.0.Wf', 'lstm_encoder.lstm.lstm_cells.0.Uf', 'lstm_encoder.lstm.lstm_cells.0.bf', 'lstm_encoder.lstm.lstm_cells.0.Wo', 'lstm_encoder.lstm.lstm_cells.0.Uo', 'lstm_encoder.lstm.lstm_cells.0.bo', 'lstm_encoder.lstm.lstm_cells.0.Wc', 'lstm_encoder.lstm.lstm_cells.0.Uc', 'lstm_encoder.lstm.lstm_cells.0.bc', 'lstm_encoder.lstm.lstm_cells.1.W_decomp', 'lstm_encoder.lstm.lstm_cells.1.b_decomp', 'lstm_encoder.lstm.lstm_cells.1.Wi', 'lstm_encoder.lstm.lstm_cells.1.Ui', 'lstm_encoder.lstm.lstm_cells.1.bi', 'lstm_encoder.lstm.lstm_cells.1.Wf', 'lstm_encoder.lstm.lstm_cells.1.Uf', 'lstm_encoder.lstm.lstm_cells.1.bf', 'lstm_encoder.lstm.lstm_cells.1.Wo', 'lstm_encoder.lstm.lstm_cells.1.Uo', 'lstm_encoder.lstm.lstm_cells.1.bo', 'lstm_encoder.lstm.lstm_cells.1.Wc', 

MultiModal(
  (lstm_encoder): TimeAwareAttentionEncoder(
    (lstm): TimeAwareLSTM(
      (lstm_cells): ModuleList(
        (0-1): 2 x TLSTMCell()
      )
    )
    (attention): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
    )
    (layer_norm_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (ff): Sequential(
      (0): Linear(in_features=512, out_features=2048, bias=True)
      (1): ReLU()
      (2): Linear(in_features=2048, out_features=512, bias=True)
    )
    (layer_norm_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (static_encoder): StaticEncoder(
    (embeddings): ModuleList(
      (0): Embedding(2, 16)
      (1): Embedding(491, 16)
      (2): Embedding(9, 16)
      (3): Embedding(2, 16)
      (4): Embedding(9, 16)
      (5): Embedding(3, 16)
    )
    (numerical_processor): Sequential(
      (0): Linear(in_features=9, out_features=16, bias=True)
      (1): ReLU()
    )
    (fc

In [4]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(SimpleMLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)  # 1 output for binary classification

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x.squeeze(-1)

In [5]:
task = "GraftLoss" 
clf_model = SimpleMLP(input_dim=512)
clf_model.load_state_dict(torch.load(f"../models/{task}@90_clf.pth", weights_only=True))
clf_model.eval()
clf_model.to(device)

SimpleMLP(
  (fc1): Linear(in_features=512, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=1, bias=True)
)

In [6]:
num_cat   = len(CONFIG['static_categorical_cols'])
num_num   = len(CONFIG['static_numerical_cols'])
F         = len(CONFIG['ts_features'])
T_max     = 90           # total padded sequence length
predict_steps_ahead = 1
T_eff = T_max - predict_steps_ahead
if T_eff <= 0:
    raise ValueError(f"T_eff={T_eff} must be > 0. Check T_max vs predict_steps_ahead.")
CONFIG['max_notes'] = 100

In [18]:
print(CONFIG)

{'static_categorical_cols': ['gender', 'underlying_disease', 'blood_group', 'gender_donor', 'donor_bloodgroup', 'type_of_donation'], 'static_numerical_cols': ['age', 'number_dialyses', 'cold_ischemia_time', 'age_donor', 'pirche_score', 'mma_broad', 'mmb_broad', 'mmdr_broad', 'mm_broad'], 'ts_features': ['bp_sys', 'bp_dia', 'weight', 'urine_volume', 'hr', 'temperature', 'diuresis_time', 'creatinine', 'leukocyte', 'proteinuria', 'crphp', 'egfr', 'acr', 'Tacrolimus', 'Methylprednisolon', 'Ciclosporin'], 'static_embedding_dim': 16, 'static_output_dim': 256, 'lstm_hidden_size': 512, 'lstm_num_layers': 2, 'num_heads': 2, 'PADDING_VAL': 0, 'notes_embedding_dim': 1024, 'max_notes': 100}


In [25]:
def pipeline_predict(static_cat, static_num, ts_feats, timesteps, mask, notes_embeddings, notes_timesteps, notes_mask):
    """
    Runs a forward pass (model -> clf_model). 
    Returns shape (B,) with probabilities.
    """
    B = static_cat.size(0)

    # Ensure the data is on the right device
    static_cat = static_cat.to(device)
    static_num = static_num.to(device)
    ts_feats   = ts_feats.to(device)
    timesteps  = timesteps.to(device)
    mask       = mask.to(device)

    notes_embeddings = notes_embeddings.to(device)
    notes_timesteps = notes_timesteps.to(device)
    notes_mask = notes_mask.to(device)

    # Make sure the feature dimensions are correct
    ts_feats = ts_feats[:, :T_eff, :]
    timesteps = timesteps[:, :T_eff]
    mask = mask[:, :T_eff]
    
    if T_eff > 1:
        intervals = timesteps[:, 1:] - timesteps[:, :-1]
    else:
        intervals = torch.zeros((B, 0), device=device)

    dummy_col = torch.zeros((B, 1), device=device, dtype=intervals.dtype)
    elapsed_times = torch.cat([intervals, dummy_col], dim=1)

    pred_ts, lstm_out, attn, static_encoding = model(
        x=ts_feats,
        elapsed_times=elapsed_times,
        timesteps=timesteps,
        static_features=(static_cat, static_num),
        mask=mask,
        notes_embeddings=notes_embeddings,
        notes_timesteps=notes_timesteps,
        notes_mask=notes_mask
    )

    final_h = lstm_out[:, -1, :]
    logits = clf_model(final_h)
    probs = torch.sigmoid(logits).detach().cpu().numpy().flatten()

    return pred_ts, lstm_out, attn, static_encoding, probs

def get_expected_feature_size():
    """Calculate the expected feature vector size."""
    return (
        num_cat +  # static categorical
        num_num +  # static numerical 
        T_eff * F +  # time series
        1  # aggregated notes mean
    )

def plot_seq(y_seq,pred, lb, ub, lens= None, range = None, color="black", label="method", ax=None):
    if ax is None:
        fig = plt.figure(figsize=(7, 4))
        ax = fig.gca()

    if range is None:
        range = (0, lens)
    time = np.arange(range[0], range[1])
    ax.plot(time, y_seq, label="True", color="black", alpha=0.5)
    ax.plot(time, pred, label="Pred", linestyle="--", color="black")

    #plot between lb and ub
    ax.fill_between(time, lb, ub, color=color, alpha=0.5, label=label)

    ax.legend(loc='lower right')


def run_analysis(test_dataloader, model, max_samples=5, T=30):
    """
    Run samples through TFN to compute something
    """
    sample_count = 0
    agg_coverage = np.zeros(shape=16)
    mean_widths = np.zeros(shape=16)
    for batch in test_dataloader:
        B = batch['static_categorical_features'].size(0)
        for i in range(B):
            if sample_count >= max_samples:
                break
            sc = batch['static_categorical_features'][i:i+1]
            sn = batch['static_numerical_features'][i:i+1]
            tf = batch['ts_features'][i:i+1]
            t = batch['timesteps'][i:i+1]
            m = batch['mask'][i:i+1]
            nt = batch['notes_embeddings'][i:i+1]
            nt_timesteps = batch['notes_timesteps'][i:i+1]
            nt_mask = batch['notes_mask'][i:i+1]
            
            sample_count += 1
            print(f"Processing sample {sample_count}/{max_samples}")

            pred_ts, lstm_out, attn, static_encoding, probs = pipeline_predict(sc, sn, tf, t, m, nt, nt_timesteps, nt_mask)

            print(f"Likelyhood of Patient suffering from {task}: {(int(probs[0] * 100))}%")

        if sample_count >= max_samples:
            break
            
    return 

In [26]:
res = run_analysis(train_dataloader, model)

Processing sample 1/5
Likelyhood of Patient suffering from GraftLoss: 17%
Processing sample 2/5
Likelyhood of Patient suffering from GraftLoss: 38%
Processing sample 3/5
Likelyhood of Patient suffering from GraftLoss: 17%
Processing sample 4/5
Likelyhood of Patient suffering from GraftLoss: 29%
Processing sample 5/5
Likelyhood of Patient suffering from GraftLoss: 3%
